# 03 — Evaluación comparativa

Carga los checkpoints entrenados y genera todas las métricas:
F1 por clase, matrices de confusión, latencia CPU y métricas temporales.

In [ ]:
import sys
sys.path.insert(0, '..')

from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import torch

from src.data.dataset import make_loaders, CLASS_NAMES
from src.training.evaluate import load_checkpoint, evaluate

## 1. Cargar test set y modelos

In [ ]:
import h5py

H5 = '/home/lilidl/drowsiness_crops.h5'
CLASS_NAMES = ['alerta', 'somnoliento']   # binario (sobreescribe el import de 3 clases)
LABEL_MAP   = {0: 0, 2: 1}

test_idx = np.load('../data/processed/test_idx.npy')

# DataLoader solo para test; train/val dummy (1 índice) para reusar la firma.
_, _, test_loader = make_loaders(
    H5, test_idx[:1], test_idx[:1], test_idx,
    batch_size=64, num_workers=8, label_map=LABEL_MAP
)

with h5py.File(H5, 'r') as f:
    y_raw = f['y'][test_idx]
y_test = np.array([LABEL_MAP[int(v)] for v in y_raw])

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_a = load_checkpoint(Path('../checkpoints/mobilenetv2_best.pt'), device=device)
model_b = load_checkpoint(Path('../checkpoints/resnet50v2_best.pt'),  device=device)

print(f'Test set: {len(y_test)} frames')
for i, name in enumerate(CLASS_NAMES):
    print(f'  {name}: {(y_test == i).sum()}')

## 2. Evaluación completa

In [ ]:
print('=== MobileNetV2 ===')
results_a = evaluate(model_a, test_loader, device=device, class_names=CLASS_NAMES)

print('\n=== ResNet50V2 ===')
results_b = evaluate(model_b, test_loader, device=device, class_names=CLASS_NAMES)

## 3. Matrices de confusión

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, results, title in [
    (axes[0], results_a, 'MobileNetV2'),
    (axes[1], results_b, 'ResNet50V2'),
]:
    cm = np.array(results['confusion_matrix'])
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    sns.heatmap(
        cm_norm, annot=cm, fmt='d', ax=ax,
        xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
        cmap='Blues', vmin=0, vmax=1,
        annot_kws={'size': 12},
    )
    ax.set_title(f'{title}\nF1-macro={results["f1_macro"]:.3f}')
    ax.set_xlabel('Predicho'); ax.set_ylabel('Real')

plt.tight_layout()
plt.savefig('../docs/confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Tabla comparativa final

In [ ]:
rows = []
for model_name, r in [('MobileNetV2', results_a), ('ResNet50V2', results_b)]:
    row = {
        'Modelo': model_name,
        'F1-macro': f"{r['f1_macro']:.4f}",
        'Latencia (ms)': f"{r['latency_cpu_ms']:.1f}",
        'FPS (CPU)': f"{1000/r['latency_cpu_ms']:.1f}",
        'Consistencia': f"{r['consistency']:.3f}",
        'Transiciones/min': f"{r['transitions_per_min']:.2f}",
    }
    for cls in CLASS_NAMES:
        row[f'F1 {cls}'] = f"{r['per_class'][cls]['f1']:.3f}"
    rows.append(row)

df = pd.DataFrame(rows).set_index('Modelo')
print(df.to_string())

df.to_csv('../docs/tabla_comparativa.csv')
print('\nTabla guardada en docs/tabla_comparativa.csv')

## 5. Dwell time por clase y modelo

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

x = np.arange(len(CLASS_NAMES))
width = 0.35

dw_a = [results_a['dwell_time_per_class'][c] for c in CLASS_NAMES]
dw_b = [results_b['dwell_time_per_class'][c] for c in CLASS_NAMES]

ax.bar(x - width/2, dw_a, width, label='MobileNetV2', color='steelblue')
ax.bar(x + width/2, dw_b, width, label='ResNet50V2',  color='tomato')
ax.set_xticks(x); ax.set_xticklabels(CLASS_NAMES)
ax.set_ylabel('Dwell time medio (s)')
ax.set_title('Tiempo de permanencia por clase')
ax.legend(); ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../docs/dwell_time.png', dpi=150, bbox_inches='tight')
plt.show()